# Basics

This section will introduce jets at a basic level. More detailed definitions are in resource links therein.

A jet (shown below) is a 'collimated spray' of particles that originate from a parton* and subsequently hadronise. Given that they are formed from the evolution of parton interactions, all the way to hadronisation and detection, jet observables can be used to study unique QCD effects, including fragmentation functions and particle hadronisation.

![A jet at LHCb](../images/jet.jpg)

There are several observables of interest, and these can be found in detail in the following resources:

- Looking inside jets: an introduction to jet substructure and boosted-object phenomenology (S. Marzani, G. Soyez & M. Spannowsky) [https://arxiv.org/pdf/1901.10342]
- Jet Physics from the Ground Up (A. Larkoski) [https://arxiv.org/pdf/2112.15122]
- Hadronic Jets (A. Barfi) [https://iopscience.iop.org/book/mono/978-1-6817-4073-7]

## Algorithms and Combination Schemes

Experimentally, the particles comprising the jet is defined by an algorithm and a particle combination schemes. Some algorithms include (but are not limited to): fixed-cone, iterative-cone and sequential algorithms. Combination schemes include (but are not limited to): $E$-scheme, $pT$-weighted scheme and WTA (Winner-Take-All) scheme.


At LHCb, a commonly used jet definition is given by the anti-$k_T$ algorithm with the $E$-scheme (addition of four-vectors). The anti-$k_T$ algorithm clusters from hardest particle first. As such it is resilient to soft-radiation, but as a result is also blind to such effects. Therefore the algorithm should be chosen carefully. A unique property of the anti-$k_T$ algorithm, compared to others such as $k_T$ ($k=1$) or Cambridge-Aachen ($k=0$), is that it is infrared and collinear (IRC) safe at all orders. This also makes it safe to analyse from a theory perspective***. The anti-$k_T$ algorithm is given by the $k=-1$ application of the following steps:

- 1. For each pair of particles $i$, $j$, define a distance 

$$d_{ij} = \min \left(p_{T_i}^{2k}, p_{T_j}^{2k}\right)\frac{\Delta R_{ij}^2}{R^2}$$

and a distance to the beam,

$$d_{iB} = p_{T_i}^{2k}$$

where $p_{T_i}$ is the transverse momentum of the particle $i$, $\Delta R_{ij}^2 = (y_i-y_j)^2+(\phi_i-\phi_j)^2$, and $R$ is the jet radius parameter**.

- 2. Find the smallest distance:
    a. If $d_{ij}$ is smallest, merge particles $i$ and $j$ into one pseudo-particle using a combination scheme (typically $E$-scheme).
    b. If $d_{iB}$ is smallest, declare particle $i$ (which may be a pseudoparticle) a jet and remove it from the list.

- 3. Repeat until no particles remain.


*: In QCD, it can be difficult to isolate one parton such as a gluon from two gluons which happen to be collinear.

**: The radius parameter, $R$, is a distance in $y-\phi$ phase space. This does not mean that the jet is circular in coordinate space. Further, jets can be 'close' to another jet, which adjusts the boundary of each jet. As a result the jet, even in phase space, need not be circular. 

***: Flavour-tagging is sometimes best done via the WTA algorithm.


## Jet Pruning

Experimentally, the jet can also undergo 'pruning' or 'grooming' processes (see below), to eliminate soft/wide radiation under certain conditions e.g.

$$\frac{\min(p_{T_1},p_{T_2})}{p_{T_1}+p_{T_2}} >_? z_{\text{cut}}\left(\frac{\Delta R_{12}}{R}\right)$$ 


More details can be found at Jets at LHCb Startertalk (E. Lesser) [https://indico.cern.ch/event/1516422/timetable/#45-jets-at-lhcb-how-we-study-t]

![An example of jet grooming.](../images/jetgroom.png)



# Simulation

## PYTHIA

Information for running jet algorithms through PYTHIA can be found in the manual [https://pythia.org/latest-manual/Welcome.html] or [https://pythia.org/latest-manual/Welcome.html]. It is worth noting that the PYTHIA simulated jets use the same code of `FastJet` (i.e the same software used to create jets at LHCb). The method for simulating jets in PYTHIA is known as `SlowJet`. As of writing, PYTHIA does not support jet grooming.

A simple example of running jet simulations in PYTHIA is below (given by `main211.cc` in PYTHIA):

```
// main211.cc is a part of the PYTHIA event generator.
// Copyright (C) 2025 Torbjorn Sjostrand.
// PYTHIA is licenced under the GNU GPL v2 or later, see COPYING for details.
// Please respect the MCnet Guidelines, see GUIDELINES for details.

// Keywords: basic usage; jet finding; slowjet;

// This is a simple test program.
// It studies jet production at the LHC, using SlowJet,
// here as a Pythia-centric wrapper for FastJet.

#include "Pythia8/Pythia.h"
using namespace Pythia8;

//==========================================================================

int main() {

  // Number of events, generated and listed ones.
  int nEvent    = 1000;
  int nListJets = 5;

  // Generator. LHC process and output selection. Initialization.
  Pythia pythia;
  pythia.readString("Beams:eCM = 14000.");
  pythia.readString("HardQCD:all = on");
  pythia.readString("PhaseSpace:pTHatMin = 200.");
  pythia.readString("Next:numberShowInfo = 0");
  pythia.readString("Next:numberShowProcess = 0");
  pythia.readString("Next:numberShowEvent = 0");

  // If Pythia fails to initialize, exit with error.
  if (!pythia.init()) return 1;

  // Standard parameters for a jet finder.
  double radius   = 0.7;
  double pTjetMin = 10.;
  double etaMax   = 4.;
  // Exclude neutrinos (and other invisible) from study.
  int    nSel     = 2;

  // Set up SlowJet jet finder, with anti-kT clustering (-1)
  // and pion mass assumed for non-photons.
  SlowJet slowJet( -1, radius, pTjetMin, etaMax, nSel, 1);

  // Histograms.
  Hist nJets("number of jets", 50, -0.5, 49.5);
  Hist pTjets("pT for jets", 100, 0., 500.);
  Hist yJets("y for jets", 100, -5., 5.);
  Hist phiJets("phi for jets", 100, -M_PI, M_PI);
  Hist distJets("R distance between jets", 100, 0., 10.);
  Hist pTdiff("pT difference between jets", 100, -100., 400.);

  // Begin event loop. Generate event. Skip if error.
  for (int iEvent = 0; iEvent < nEvent; ++iEvent) {
    if (!pythia.next()) continue;

    // Analyze Slowet jet properties. List first few.
    slowJet.analyze( pythia.event );
    if (iEvent < nListJets) slowJet.list();

    // Fill SlowJet inclusive jet distributions.
    nJets.fill( slowJet.sizeJet() );
    for (int i = 0; i < slowJet.sizeJet(); ++i) {
      pTjets.fill( slowJet.pT(i) );
      yJets.fill( slowJet.y(i) );
      phiJets.fill( slowJet.phi(i) );
    }

    // Fill SlowJet distance between jets.
    for (int i = 0; i < slowJet.sizeJet() - 1; ++i)
    for (int j = i + 1; j < slowJet.sizeJet(); ++j) {
      double dY = slowJet.y(i) - slowJet.y(j);
      double dPhi = abs( slowJet.phi(i) - slowJet.phi(j) );
      if (dPhi > M_PI) dPhi = 2. * M_PI - dPhi;
      double dR = sqrt( pow2(dY) + pow2(dPhi) );
      distJets.fill( dR );
    }

    // Fill SlowJet pT-difference between jets (to check ordering of list).
    for (int i = 1; i < slowJet.sizeJet(); ++i)
      pTdiff.fill( slowJet.pT(i-1) - slowJet.pT(i) );

  // End of event loop. Statistics. Histograms.
  }
  pythia.stat();
  cout << nJets << pTjets << yJets << phiJets << distJets << pTdiff;

  // Done.
  return 0;
}

```
The user can print out some jet statistics via `slowJet.list()`, whose output is shown below:


```
 --  PYTHIA SlowJet(fjcore) Listing, p = -1, R = 0.500, pTjetMin =   6.000, etaMax = 25.000  -- 
 
   no      pTjet      y       phi   mult      p_x        p_y        p_z         e          m 
    0     10.031   -2.074    1.423    15      1.478      9.922    -41.889     43.235      3.730
    1      9.612   -2.002   -2.725     9     -8.789     -3.892    -37.370     38.759      3.646
    2      7.733    0.824   -1.268    15      2.305     -7.382      8.335     12.307      4.711
    3      6.917    3.562    0.625     4      5.611      4.046    125.257    125.459      1.672
    4      6.400   -2.552    0.363    10      5.984      2.272    -42.116     42.630      1.614
```

It is also possible to examine the particles within the jet:

```
for(int j = 0; j < slowJet.sizeJet(); j++){
    \\Loop through all jets in an event

    for(size_t k: slowJet.constituents(j)){
        \\Loop through all constituents of a given jet

        \\ Insert whatever information you'd like to know about particle, `k`, in jet, `j`, here. You can use `pythia.event[k].index()` or other such functions as normal.
    }
}
```

## Gauss

No idea, yet.

# Analysis Production

This section demonstrates how to produce jets using the Analsyis Productions framework. As of writing this is valid for the Run 3 data-taking period at LHCb.

## Key Functions/Imports

## ParticleFlow

## Useful Functors